In [ ]:
%pip install pandas numpy

# Pandas - Data Wrangling Handbook
Series, DataFrames, cleaning, groupby, merging, time-series, and method chaining.

In [ ]:
import pandas as pd
import numpy as np

print('Pandas version:', pd.__version__)
pd.set_option('display.max_columns', 12)
pd.set_option('display.width', 90)

## 1. Series

In [ ]:
s = pd.Series([10, 20, 30, 40, 50], index=['a', 'b', 'c', 'd', 'e'])
print(s)
print('\nDtype:', s.dtype)
print('Index:', s.index.tolist())
print('Values > 25:\n', s[s > 25])
print('Cumsum:\n', s.cumsum())

## 2. DataFrame Creation

In [ ]:
rng = np.random.default_rng(42)

df = pd.DataFrame({
    'name':       ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank'],
    'department': ['Eng', 'Mkt', 'Eng', 'HR', 'Mkt', 'Eng'],
    'salary':     [95000, 72000, 105000, 68000, 80000, 115000],
    'years':      [5, 3, 8, 2, 4, 10],
    'score':      rng.uniform(60, 100, 6).round(1)
})
df

## 3. Inspection Methods

In [ ]:
print(df.shape)
print()
df.info()

In [ ]:
df.describe()

## 4. Selection - loc / iloc

In [ ]:
# Label-based
print(df.loc[0:2, ['name', 'salary']])
print()
# Position-based
print(df.iloc[1:4, 0:3])

In [ ]:
# Boolean filter
print(df[(df['salary'] > 80000) & (df['department'] == 'Eng')])

## 5. Adding, Modifying, and Dropping Columns

In [ ]:
df2 = df.copy()
df2['bonus']    = (df2['salary'] * 0.1).round(0).astype(int)
df2['grade']    = pd.cut(df2['score'], bins=[0, 70, 85, 100],
                         labels=['C', 'B', 'A'])
df2 = df2.drop(columns=['score'])
df2

## 6. Missing Data

In [ ]:
messy = df.copy().astype(object)
messy.loc[[1, 4], 'salary'] = np.nan
messy.loc[2, 'score']       = np.nan

print('Missing counts:\n', messy.isnull().sum())
print()

# Fill numeric NaN with column median
for col in ['salary', 'score']:
    messy[col] = messy[col].fillna(messy[col].median())

print('After fill:\n', messy.isnull().sum())

## 7. GroupBy and Aggregation

In [ ]:
# Single agg
print(df.groupby('department')['salary'].mean().round(0))
print()

# Multiple agg functions
agg_result = df.groupby('department').agg(
    headcount=('name', 'count'),
    avg_salary=('salary', 'mean'),
    max_years=('years', 'max'),
    avg_score=('score', 'mean')
).round(1)
agg_result

## 8. Merging / Joining

In [ ]:
dept_info = pd.DataFrame({
    'department': ['Eng', 'Mkt', 'HR'],
    'location':   ['NYC', 'LA', 'Chicago'],
    'budget':     [500_000, 300_000, 200_000]
})

merged = df.merge(dept_info, on='department', how='left')
merged

## 9. Pivot Tables

In [ ]:
sales_data = pd.DataFrame({
    'month':   ['Jan', 'Jan', 'Feb', 'Feb', 'Mar', 'Mar'],
    'product': ['A', 'B', 'A', 'B', 'A', 'B'],
    'revenue': [120, 90, 140, 110, 160, 95]
})

pivot = sales_data.pivot_table(values='revenue', index='month',
                               columns='product', aggfunc='sum')
pivot

## 10. Time-Series

In [ ]:
dates = pd.date_range('2024-01-01', periods=90, freq='D')
rng_ts = np.random.default_rng(99)
ts = pd.Series(
    100 + np.cumsum(rng_ts.standard_normal(90)),
    index=dates,
    name='price'
)

print('First 5:\n', ts.head())
print('\n7-day rolling mean (first 10):\n', ts.rolling(7).mean().head(10))
print('\nMonthly mean:\n', ts.resample('ME').mean().round(2))

## 11. String Methods

In [ ]:
text_col = pd.Series(['  hello world  ', 'FOO BAR', 'pandas is great', 'PYTHON 3.10'])

print(text_col.str.strip().str.title())
print()
print(text_col.str.lower().str.contains('python'))

## 12. Method Chaining

In [ ]:
result = (
    df
    .query("salary > 70000")
    .assign(salary_k=lambda d: (d['salary'] / 1000).round(1))
    .sort_values('salary_k', ascending=False)
    .reset_index(drop=True)
    [['name', 'department', 'salary_k', 'years']]
)
result